# 6. Analysis

## 6.2 Material outcome
answer the computational research question. Did the system work? This covers ES convergence behaviour (did fitness improve over generations, how many evaluations to convergence), MILP assignment statistics (how many feasible pairings per slot, solve time), and GNN prediction reliability on the optimised configuration versus the Karamba3D ground truth. This is where you demonstrate the pipeline is technically valid before showing what it produced
what i mean is that you compare an optimized GA geometry of stock A, which i can already train / have and a static grid of vertices and edges and run that one time. this grid would be the normal of the c00_headquarter_params, so with grid distance of 3 meters and 5x3 cells, that would give 6x4 even distribution of 3 meters on top and 5x3 even distribution on the lower grid, and all lower grid have a z value - layerheight of top layer, this creates a very static grid, maybe lets start with creating the grid


In [ ]:
# =============================================================================
# Build static regular-grid space frame
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from collections import Counter
import importlib
import c00_headquarter_params as c11_params

importlib.reload(stage_geometry)

# ── Build ──────────────────────────────────────────────────────────────────────
static_geo = stage_geometry.run_geometry_from_design({}, sample_id=0)
df_v_s     = static_geo["df_vertices"]
df_e_s     = static_geo["df_edges"]
df_geo_s   = static_geo["df_geometry_overview"]

# ── Classify each member by the layer of its two endpoints ────────────────────
_vlu_s = (
    df_v_s
    .set_index("vertex_index")[["x", "y", "z", "layer", "attribute"]]
    .to_dict("index")
)

def _vk(v):
    s = str(v)
    return s if s.startswith("v") else f"v{s}"

_mtypes = []
for _, e in df_e_s.iterrows():
    l1 = _vlu_s[_vk(e["V1"])]["layer"]
    l2 = _vlu_s[_vk(e["V2"])]["layer"]
    _mtypes.append(
        "top_chord" if (l1 == l2 == "top")
        else "bot_chord" if (l1 == l2 == "bottom")
        else "web"
    )
df_geo_s["member_type"] = _mtypes
_ct = Counter(_mtypes)

# ── Print statistics ──────────────────────────────────────────────────────────
n_top = int((df_v_s["layer"] == "top").sum())
n_bot = int((df_v_s["layer"] == "bottom").sum())

print(f"Static regular-grid space frame")
print(f"  Grid:    {c11_params.GRID_CELLS_X} × {c11_params.GRID_CELLS_Y} cells  |  "
      f"edge = {c11_params.EDGE_LENGTH} m  |  layer h = {c11_params.LAYER_HEIGHT} m")
print(f"  Nodes:   {len(df_v_s)}  ({n_top} top,  {n_bot} bottom)")
print(f"  Members: {len(df_e_s)}  "
      f"(top chord: {_ct['top_chord']},  "
      f"bottom chord: {_ct['bot_chord']},  "
      f"web: {_ct['web']})")

print(f"\n  Member lengths  (metres)")
print(f"  {'Type':<16} {'min':>6}  {'mean':>6}  {'max':>6}  {'std':>6}")
print(f"  {'-'*44}")
for mt, lbl in [("top_chord","Top chord"), ("bot_chord","Bot chord"), ("web","Web/diagonal")]:
    s = df_geo_s[df_geo_s["member_type"] == mt]["length_m"]
    print(f"  {lbl:<16} {s.min():>6.3f}  {s.mean():>6.3f}  {s.max():>6.3f}  {s.std():>6.3f}")
all_l = df_geo_s["length_m"]
print(f"  {'All':<16} {all_l.min():>6.3f}  {all_l.mean():>6.3f}  {all_l.max():>6.3f}  {all_l.std():>6.3f}")

top_z = df_v_s[df_v_s["layer"] == "top"]["z"].iloc[0]
bot_z = df_v_s[df_v_s["layer"] == "bottom"]["z"].iloc[0]
print(f"\n  z top layer:    {top_z:.3f} m  (after centring)")
print(f"  z bottom layer: {bot_z:.3f} m  (after centring)")

# ── Colour palette ────────────────────────────────────────────────────────────
_C = dict(
    top_chord="#1f77b4",
    bot_chord="#ff7f0e",
    web      ="#bbbbbb",
    support  ="#d62728",
    load     ="#2ca02c",
    bottom_n ="#9467bd",
)
_MSTYLE = dict(
    top_chord=dict(color=_C["top_chord"], lw=2.0, alpha=0.95),
    bot_chord=dict(color=_C["bot_chord"], lw=2.0, alpha=0.95),
    web      =dict(color=_C["web"],       lw=0.85, alpha=0.45),
)

# ── Pre-build segment arrays, grouped by member type ─────────────────────────
_v1k = df_e_s["V1"].map(lambda v: v if str(v).startswith("v") else f"v{v}")
_v2k = df_e_s["V2"].map(lambda v: v if str(v).startswith("v") else f"v{v}")
_all_segs = np.array([
    [[_vlu_s[v1]["x"], _vlu_s[v1]["y"], _vlu_s[v1]["z"]],
     [_vlu_s[v2]["x"], _vlu_s[v2]["y"], _vlu_s[v2]["z"]]]
    for v1, v2 in zip(_v1k, _v2k)
])  # (n_edges, 2, 3)
_mt_arr = np.array(_mtypes)

# ── Node masks (compute once, reuse per subplot) ──────────────────────────────
_sup_m = df_v_s["attribute"] == "support"
_bot_m = (~_sup_m) & (df_v_s["layer"] == "bottom")
_top_m = (~_sup_m) & (df_v_s["layer"] == "top")

_xs = df_v_s["x"].values
_ys = df_v_s["y"].values
_zs = df_v_s["z"].values

# ── Figure: perspective  +  top view ─────────────────────────────────────────
fig_static, (ax_p, ax_t) = plt.subplots(
    1, 2, figsize=(14, 6),
    subplot_kw={"projection": "3d"},
)
fig_static.suptitle(
    f"Static Regular Space Frame  —  "
    f"{c11_params.GRID_CELLS_X}×{c11_params.GRID_CELLS_Y} grid  |  "
    f"edge = {c11_params.EDGE_LENGTH} m  |  h = {c11_params.LAYER_HEIGHT} m  |  "
    f"{len(df_v_s)} nodes  ·  {len(df_e_s)} members  "
    f"({_ct['top_chord']} top  /  {_ct['bot_chord']} bot  /  {_ct['web']} webs)",
    fontsize=10.5, fontweight="bold",
)

for ax, elev, azim, title in [
    (ax_p, 26, -50, "Perspective"),
    (ax_t, 85, -90, "Top view  (looking down)"),
]:
    for mt, style in _MSTYLE.items():
        ax.add_collection3d(Line3DCollection(_all_segs[_mt_arr == mt], **style))

    for mask, col, ms in [(_sup_m, _C["support"], 55), (_bot_m, _C["bottom_n"], 28), (_top_m, _C["load"], 22)]:
        sub = df_v_s[mask]
        ax.scatter3D(sub["x"].values, sub["y"].values, sub["z"].values, c=col, s=ms, zorder=5)

    ax.auto_scale_xyz(_xs, _ys, _zs)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("x [m]", fontsize=8)
    ax.set_ylabel("y [m]", fontsize=8)
    ax.set_zlabel("z [m]", fontsize=8, labelpad=10)
    ax.set_box_aspect((1, 0.65, 0.28))
    ax.view_init(elev=elev, azim=azim)
    ax.tick_params(labelsize=7)

_leg = [
    mpatches.Patch(color=_C["top_chord"],
                   label=f"Top chord  ({_ct['top_chord']} members,  3 m each)"),
    mpatches.Patch(color=_C["bot_chord"],
                   label=f"Bottom chord  ({_ct['bot_chord']} members)"),
    mpatches.Patch(color=_C["web"],
                   label=f"Web / diagonal  ({_ct['web']} members)"),
    plt.Line2D([], [], marker="o", color="w",
               markerfacecolor=_C["support"], markersize=9,
               label=f"Support node  (4 corners)"),
    plt.Line2D([], [], marker="o", color="w",
               markerfacecolor=_C["load"], markersize=7,
               label=f"Top load node  ({n_top - 4})"),
    plt.Line2D([], [], marker="o", color="w",
               markerfacecolor=_C["bottom_n"], markersize=7,
               label=f"Bottom node  ({n_bot})"),
]
ax_p.legend(handles=_leg, loc="upper left", fontsize=7.5,
            framealpha=0.9, bbox_to_anchor=(-0.05, 1.10))

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

In [ ]:
# Export static grid to 01_geometry_data for Grasshopper inspection
df_vertices_static = df_v_s.copy()
df_edges_static    = df_e_s.copy()
df_edges_static.insert(0, "sample_id", 0)

_out = config.GEOM_DATA_PATH
_out.mkdir(parents=True, exist_ok=True)

_vpath = _out / "static_grid_vertices.csv"
_epath = _out / "static_grid_edges.csv"

df_vertices_static.to_csv(_vpath, index=False)
df_edges_static.to_csv(_epath, index=False)

print(f"Saved: {_vpath}")
print(f"Saved: {_epath}")
print(f"  Vertices: {len(df_vertices_static)} rows  {list(df_vertices_static.columns)}")
print(f"  Edges:    {len(df_edges_static)} rows  {list(df_edges_static.columns)}")

Saved: C:\Users\VR Guest\OneDrive\06 Building Technology TU\2.2 - 2.4\30_Data_Inventory\01_geometry_data\static_grid_vertices.csv
Saved: C:\Users\VR Guest\OneDrive\06 Building Technology TU\2.2 - 2.4\30_Data_Inventory\01_geometry_data\static_grid_edges.csv
  Vertices: 39 rows  ['sample_id', 'vertex_index', 'layer', 'attribute', 'x', 'y', 'z']
  Edges:    120 rows  ['sample_id', 'edge_id', 'V1', 'V2']


## 6.3 Baseline comparison: static grid vs GA-optimised design
Runs the static regular grid through the **exact same pipeline** (feasibility → cost matrix → MILP → GNN → fitness) as `GA_A_20260520_105730_RUN10_GEN250_EVAL7500_F-2_5262`, using identical stock (A), normalisation constants, fitness weights, and GNN model. The delta quantifies the material-outcome gain attributable to geometry adaptation alone.

In [1]:
# =============================================================================
# Static grid vs GA_A — single-shot pipeline comparison
# =============================================================================
# Everything needed is loaded here so the cell is self-contained.
# The static grid uses params={} which triggers all-default vertex positions
# (top nodes on exact 3 m grid, bottom nodes at cell centres, no z-shift).
# =============================================================================

import json
import importlib
import pandas as pd
from pathlib import Path

import config
from workflows import c22_stage_geometry   as stage_geometry
from workflows import c27_stage_GNN        as stage_gnn
from workflows import c23_ga_evaluator     as ga_eval

for _m in [stage_geometry, stage_gnn, ga_eval]:
    importlib.reload(_m)

# ── Reference GA run ──────────────────────────────────────────────────────────
_GA_STEM = "GA_A_20260520_105730_RUN10_GEN250_EVAL7500_F-2_5262"
_GA_DIR  = config.GA_DATA_PATH / _GA_STEM

with open(_GA_DIR / f"{_GA_STEM}_run_config.json", encoding="utf-8") as f:
    _rc = json.load(f)
with open(_GA_DIR / f"{_GA_STEM}_best_design.json", encoding="utf-8") as f:
    _best = json.load(f)

_NORM    = _rc["normalization_constants"]   # C_max / R_max as used during the GA run
_MPREFIX = _rc["model_prefix"]
_GC      = _rc["ga_config"]

_GA_CONFIG_STATIC = {
    "fitness_weights":    _GC["fitness_weights"],
    "new_stock_max_uses": _GC["new_stock_max_uses"],
    "min_reuse_fraction": _GC["min_reuse_fraction"],
    "penalty_fitness":    _GC["penalty_fitness"],
    "w_structural_start": _GC["w_structural_start"],
    "w_structural_end":   _GC["w_structural_end"],
    "use_gnn":            _GC["use_gnn"],
}

print(f"GA run  : {_GA_STEM}")
print(f"Stock   : {_rc['stock']['source']}  ({_rc['stock']['n_total']} elements,  "
      f"NS={_rc['stock']['n_ns']}  RS={_rc['stock']['n_rs']})")
print(f"Norm    : C_max={_NORM['C_max']:.4f}   R_max={_NORM['R_max']:.4f}")
print(f"Weights : ω1={_GC['fitness_weights']['omega_1']}  "
      f"ω2={_GC['fitness_weights']['omega_2']}  "
      f"ω4={_GC['w_structural_end']}")

# ── Load stock A ──────────────────────────────────────────────────────────────
_stock_path = config.TIMBER_STOCK_PATH / "complete_timber_A.csv"
df_stock_A = None
for _opts in [{"sep": ";", "encoding": "utf-8"}, {"sep": ",", "encoding": "utf-8"},
              {"sep": ";", "encoding": "latin1"}, {"sep": ",", "encoding": "latin1"}]:
    try:
        _df = pd.read_csv(_stock_path, **_opts)
        if _df.shape[1] > 1:
            df_stock_A = _df
            break
    except Exception:
        pass
df_stock_A.columns = df_stock_A.columns.str.strip()
_prep_gnn_A = stage_gnn.prepare_stock_for_gnn(df_stock_A)
print(f"Stock A loaded: {len(df_stock_A)} elements\n")

# ── Run static grid through the full pipeline ────────────────────────────────
print("─" * 62)
print("Running static grid  (params={})  through full pipeline …")
print("─" * 62)

_static_eval = ga_eval.evaluate_design_candidate(
    design_params        = {},       # empty → all-default regular grid
    df_stock             = df_stock_A,
    fixed_norm_constants = _NORM,
    config_dict          = _GA_CONFIG_STATIC,
    model_prefix         = _MPREFIX,
    generation           = 250,      # end-of-run gen → w_structural = 0.8
    max_generations      = 250,
    sample_id            = 0,
    verbose              = True,
    prepared_gnn_stock   = _prep_gnn_A,
)

print(f"\nStatic eval  →  status: {_static_eval['status']}")
if _static_eval.get("reason"):
    print(f"  reason: {_static_eval['reason']}")

# ── Comparison table ──────────────────────────────────────────────────────────
_sfr = _static_eval.get("fitness_result", {})
_gfr = _best.get("fitness_result", {})

_metrics = [
    ("Fitness",                   _sfr.get("fitness",                  float("nan")), _best["fitness"]),
    ("Total cost  (kg CO₂e)",     _static_eval.get("total_cost",       float("nan")), _best["total_cost"]),
    ("Reuse fraction",            _static_eval.get("reuse_fraction",   float("nan")), _best["reuse_rate"]),
    ("GNN feasibility",           _static_eval.get("gnn_feasibility",  float("nan")), _best["gnn_feasibility"]),
    ("cost_norm",                 _sfr.get("cost_norm",                float("nan")), _gfr.get("cost_norm",  float("nan"))),
    ("reuse_norm",                _sfr.get("reuse_norm",               float("nan")), _gfr.get("reuse_norm", float("nan"))),
    ("structural infeasibility",  _sfr.get("structural_infeasibility", float("nan")), _gfr.get("structural_infeasibility", float("nan"))),
    ("MILP status",               _static_eval.get("milp_status", "—"),              _best["milp_status"]),
]

_rows = []
for label, sv, gv in _metrics:
    if isinstance(sv, float) and isinstance(gv, float):
        delta_str = f"{gv - sv:+.4f}"
    else:
        delta_str = "—"
    _rows.append({
        "Metric":           label,
        "Static grid":      f"{sv:.4f}" if isinstance(sv, float) else sv,
        f"GA best ({_GA_STEM[:4]})": f"{gv:.4f}" if isinstance(gv, float) else gv,
        "Δ  (GA − static)": delta_str,
    })

df_comparison = pd.DataFrame(_rows).set_index("Metric")

print(f"\n{'='*62}")
print(f"STATIC GRID  vs  GA-OPTIMISED  (stock A,  identical pipeline)")
print(f"{'='*62}")
display(df_comparison)

Config System loaded successfully, Code running locally from thesis_generative_timber and Data is connected to OneDrive 2.2 - 2.4.

Grid: 5x3, edge=3.0 m, height=1.5 m, divisions=8, samples=20000
LCA factors: A1-A3=0.25, C1=0.0085, A5 prep=0.01, A5 saw=0.004, C2 dist=50 km, C3-C4=0.031, ω=0


c:\Users\VR Guest\Documents\PyRepo\thesis_generative_timber\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GA run  : GA_A_20260520_105730_RUN10_GEN250_EVAL7500_F-2_5262
Stock   : C:\Users\VR Guest\OneDrive\06 Building Technology TU\2.2 - 2.4\30_Data_Inventory\03_timber_data\complete_timber_A.csv  (524 elements,  NS=421  RS=103)
Norm    : C_max=384.8784   R_max=0.5123
Weights : ω1=1.0  ω2=3.0  ω4=0.8
Stock A loaded: 524 elements

──────────────────────────────────────────────────────────────
Running static grid  (params={})  through full pipeline …
──────────────────────────────────────────────────────────────
    ✓ geometry    | 39 nodes, 120 edges
  Stage 1 (length):    38,460 eliminated  (24,420 remaining, 38.8%)
  Stage 2 (force):   max tension=12.0 kN  max compression=-10.4 kN  mean |F|=4.5 kN
  Stage 3 (EC5):        3,980 eliminated  (20,440 remaining, 32.5%)
    ✓ feasibility | 20,440 feasible slot/stock pairs
    ✓ cost matrix | 20,440 finite entries
    ✓ MILP        | status=Optimal, cost=117.1860, 120 assignments
[IO] Inference config loaded from ID20260516_182257_LR1e-04_EP200_BS

,Static grid,GA best (GA_A),Δ (GA − static)
Metric,,,
Fitness,-1.1957,-2.5262,-1.3305
Total cost (kg CO₂e),117.1860,111.0419,-6.1441
Reuse fraction,0.3026,0.5490,+0.2464
GNN feasibility,0.6605,0.7684,+0.1080
cost_norm,0.3045,0.2885,-0.0160
reuse_norm,0.5906,1.0000,+0.4094
structural infeasibility,0.3395,0.2316,-0.1080
MILP status,Optimal,Optimal,—


## 6.4 Cross-run comparison: GA_new / GA_A / GA_B
Full hard-metric comparison across stock scenarios. Normalisation bounds differ between runs and are excluded from the primary comparison, but listed in the summary for reference.

In [ ]:
# =============================================================================
# 6.4  Cross-run comparison — GA_new / GA_A / GA_B
# =============================================================================
# Edit RUN_STEMS to swap in different runs. Everything else is automatic.
# Hard values only — normalisation bounds differ across runs and are listed
# for reference only, not used in the primary comparison.
# =============================================================================

import json
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from pathlib import Path

import config
importlib.reload(config)

from workflows import c22_stage_geometry      as stage_geometry
from workflows import c24_stage_feasibility   as stage_feas
from workflows import c28_stage_fitness_score as stage_fitness

for _m in [stage_geometry, stage_feas, stage_fitness]:
    importlib.reload(_m)

# Pull the project colour palette once
_PC  = config.PLOT_COLORS
_EXT = _PC["extra_colors"]

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — set stems here
# ─────────────────────────────────────────────────────────────────────────────
RUN_STEMS = {
    "new": "GA_new_20260519_180633_GEN250_EVAL7500_F0_7394",
    "A":   "GA_A_20260520_105730_RUN10_GEN250_EVAL7500_F-2_5262",
    "B":   "GA_B_20260520_132227_GEN250_EVAL7500_F-2_2732",
}

# Per-run colours (box plots / labels)
RUN_COLORS = {
    "new": _EXT["muted_teal"],
    "A":   _EXT["soft_sage_green"],
    "B":   _EXT["dusty_plum"],
}

# Member material colours
_C_RS  = _PC["RS"]           # #F2994B  reclaimed (orange)
_C_NS  = _PC["NS"]           # #61788C  new stock (blue)

# Node colours
_C_SUP  = _PC["upper_node"]  # #2F3E4F  supports (dark navy)
_C_LOAD = _PC["black"]       # #000000  top load nodes
_C_BOT  = _PC["lower_node"]  # #D9653B  bottom nodes

# ─────────────────────────────────────────────────────────────────────────────
# LOAD artefacts for each run
# ─────────────────────────────────────────────────────────────────────────────
runs = {}
for label, stem in RUN_STEMS.items():
    d = config.GA_DATA_PATH / stem
    with open(d / f"{stem}_run_config.json",  encoding="utf-8") as f: rc = json.load(f)
    with open(d / f"{stem}_best_design.json", encoding="utf-8") as f: bd = json.load(f)
    tk = d / "top_k_designs"
    runs[label] = dict(
        stem       = stem,
        rc         = rc,
        bd         = bd,
        tk_summary = pd.read_csv(tk / f"{stem}_top10_summary.csv"),
        tk_verts   = pd.read_csv(tk / f"{stem}_top10_vertices.csv"),
        tk_edges   = pd.read_csv(tk / f"{stem}_top10_edges_assigned.csv"),
    )
    print(f"  {label:>3}  {stem}"
          f"\n       fitness={bd['fitness']:.4f}  reuse={bd['reuse_rate']:.1%}"
          f"  cost={bd['total_cost']:.1f} kg CO₂e\n")

# ─────────────────────────────────────────────────────────────────────────────
# WASTE CALCULATION — re-runs geometry + feasibility per best design
# ─────────────────────────────────────────────────────────────────────────────
def _load_stock_df(stock_source: str) -> pd.DataFrame:
    p = Path(stock_source)
    for opts in [{"sep": ";", "encoding": "utf-8"},
                 {"sep": ",", "encoding": "utf-8"},
                 {"sep": ";", "encoding": "latin1"},
                 {"sep": ",", "encoding": "latin1"}]:
        try:
            df = pd.read_csv(p, **opts)
            if df.shape[1] > 1:
                df.columns = df.columns.str.strip()
                return df
        except Exception:
            pass
    raise RuntimeError(f"Could not load stock from {p}")


def _compute_waste_for_run(r: dict) -> float:
    """Re-run geometry + feasibility for the best design, return offcut waste (m³)."""
    design_params = r["bd"].get("design_params", {})
    df_stock      = _load_stock_df(r["rc"]["stock"]["source"])

    geo_out  = stage_geometry.run_geometry_from_design(design_params, sample_id=0)
    df_edges = geo_out["df_edges"]
    df_verts = geo_out["df_vertices"].copy()
    df_verts["v_idx"] = df_verts["vertex_index"].str.replace("v", "", regex=False).astype(int)
    df_verts = df_verts.sort_values("v_idx").reset_index(drop=True)

    node_positions = df_verts[["x", "y", "z"]].values
    support_nodes  = df_verts[df_verts["attribute"] == "support"]["v_idx"].tolist()
    load_nodes     = df_verts[df_verts["attribute"] == "load"]["v_idx"].tolist()

    df_slots, _, _, _ = stage_feas.build_cost_filter(
        node_positions = node_positions,
        edges_df       = df_edges,
        stock_df       = df_stock,
        support_nodes  = support_nodes,
        load_nodes     = load_nodes,
        verbose        = False,
    )

    milp_df = r["tk_edges"][r["tk_edges"]["rank"] == 1][["edge_id", "assigned_timber"]].copy()
    return stage_fitness.calculate_total_waste(milp_df, df_slots, df_stock)


print("Computing offcut waste for each run …")
for label in RUN_STEMS:
    try:
        waste = _compute_waste_for_run(runs[label])
        runs[label]["waste_m3"] = waste
        print(f"  GA_{label}: {waste:.4f} m³")
    except Exception as exc:
        runs[label]["waste_m3"] = float("nan")
        print(f"  GA_{label}: ERROR — {exc}")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION A — Hard-metric summary table
# ─────────────────────────────────────────────────────────────────────────────
_metrics = [
    # objective
    ("Fitness (objective)",          lambda r: r["bd"]["fitness"]),
    # cost
    ("Total cost  (kg CO₂e)",        lambda r: r["bd"]["total_cost"]),
    ("Avg cost per member  (CO₂e)",  lambda r: r["bd"]["total_cost"] / 120),
    # reuse / material
    ("Reuse fraction",               lambda r: r["bd"]["reuse_rate"]),
    ("Reclaimed slots  (of 120)",    lambda r: round(r["bd"]["reuse_rate"] * 120)),
    # waste
    ("Offcut waste  (m³)",           lambda r: r["waste_m3"]),
    # structural
    ("GNN feasibility",              lambda r: r["bd"]["gnn_feasibility"]),
    ("Unsafe members  (GNN)",        lambda r: len(r["bd"]["gnn_unsafe_members"])),
    ("ω_structural applied",         lambda r: r["bd"]["w_structural"]),
    # optimisation run
    ("Best found — generation",      lambda r: r["bd"]["generation"]),
    ("MILP status",                  lambda r: r["bd"]["milp_status"]),
    ("GA evaluations",               lambda r: r["rc"]["n_evals"]),
    # stock pool
    ("Stock — total elements",       lambda r: r["rc"]["stock"]["n_total"]),
    ("Stock — NS",                   lambda r: r["rc"]["stock"]["n_ns"]),
    ("Stock — RS",                   lambda r: r["rc"]["stock"]["n_rs"]),
    ("RS availability  (%)",         lambda r: r["rc"]["stock"]["n_rs"] / r["rc"]["stock"]["n_total"]),
    # norm bounds — reference only
    ("C_max  [norm ref]",            lambda r: r["rc"]["normalization_constants"]["C_max"]),
    ("R_max  [norm ref]",            lambda r: r["rc"]["normalization_constants"]["R_max"]),
]

tbl = {}
for label in RUN_STEMS:
    col = {}
    for name, fn in _metrics:
        try:   col[name] = fn(runs[label])
        except Exception: col[name] = "—"
    tbl[f"GA_{label}"] = col

df_tbl = pd.DataFrame(tbl).rename_axis("Metric")

def _fmt(v):
    if isinstance(v, float): return f"{v:.4f}"
    if isinstance(v, int):   return str(v)
    return v

df_display = df_tbl.apply(lambda col: col.map(_fmt))

print(f"\n{'='*58}\nHARD-METRIC SUMMARY\n{'='*58}")
display(df_display)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION B — Top-10 distribution box plots
# ─────────────────────────────────────────────────────────────────────────────
_box_defs = [
    ("fitness",         "Fitness",              None),
    ("total_cost",      "Total cost (kg CO₂e)", None),
    ("reuse_rate",      "Reuse fraction",        (0, 1)),
    ("gnn_feasibility", "GNN feasibility",       (0, 1)),
]

fig_box, axes_b = plt.subplots(1, 4, figsize=(16, 4.5))
fig_box.suptitle("Top-10 solution distributions  (GA_new / GA_A / GA_B)",
                 fontweight="bold", fontsize=11)

labels_list = list(RUN_STEMS.keys())
colors_list = [RUN_COLORS[l] for l in labels_list]

for ax, (col, title, ylim) in zip(axes_b, _box_defs):
    data  = [runs[l]["tk_summary"][col].dropna().values for l in labels_list]
    bp = ax.boxplot(data, patch_artist=True, widths=0.45,
                    medianprops=dict(color=_PC["black"], lw=2))
    for patch, color in zip(bp["boxes"], colors_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.72)
    for i, (d, color) in enumerate(zip(data, colors_list), start=1):
        ax.scatter(np.full(len(d), i) + np.random.uniform(-0.08, 0.08, len(d)),
                   d, color=color, s=22, alpha=0.88, zorder=5)
    ax.set_xticks(range(1, len(labels_list) + 1))
    ax.set_xticklabels([f"GA_{l}" for l in labels_list], fontsize=9)
    ax.set_title(title, fontsize=9.5, fontweight="bold")
    ax.tick_params(labelsize=8)
    ax.grid(axis="y", linestyle="--", alpha=config.PLOT_STYLE["grid_alpha"])
    if ylim:
        ax.set_ylim(*ylim)

plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# SECTION C — Best-design geometry: perspective + top view, RS/NS coloured
# ─────────────────────────────────────────────────────────────────────────────
n_runs = len(RUN_STEMS)

fig_geo, axes_geo = plt.subplots(
    2, n_runs,
    figsize=(6.0 * n_runs, 11),
    subplot_kw={"projection": "3d"},
)

fig_geo.suptitle(
    "Best-design geometry — orange = reclaimed RS  ·  blue = new NS",
    fontweight="bold", fontsize=11,
)

_VIEWS = [
    (26, -50, "Perspective"),
    (88, -90, "Top view"),
]

for col_idx, label in enumerate(RUN_STEMS):
    r   = runs[label]
    bd  = r["bd"]

    vdf = r["tk_verts"][r["tk_verts"]["rank"] == 1].copy()
    vdf["vi"] = vdf["vertex_index"].str[1:].astype(int)
    vlookup = vdf.set_index("vi")[["x", "y", "z", "layer", "attribute"]]

    edf = r["tk_edges"][r["tk_edges"]["rank"] == 1].copy()
    rs_count = edf["assigned_timber"].astype(str).str.startswith("RS").sum()

    # ── Pre-build edge segment arrays (RS and NS) ──────────────────────────
    v1s = edf["V1"].astype(int).values
    v2s = edf["V2"].astype(int).values
    valid = np.isin(v1s, vlookup.index) & np.isin(v2s, vlookup.index)
    v1s, v2s = v1s[valid], v2s[valid]
    is_rs_arr = edf["assigned_timber"].astype(str).str.startswith("RS").values[valid]

    p1_coords = vlookup.loc[v1s, ["x", "y", "z"]].values
    p2_coords = vlookup.loc[v2s, ["x", "y", "z"]].values
    segs = np.stack([p1_coords, p2_coords], axis=1)  # (n_valid, 2, 3)

    # ── Node masks (compute once per run) ─────────────────────────────────
    _sup_v = vdf["attribute"] == "support"
    _bot_v = (~_sup_v) & (vdf["layer"] == "bottom")
    _top_v = ~_sup_v & ~_bot_v
    _vxyz  = vdf[["x", "y", "z"]].values

    for row_idx, (elev, azim, view_lbl) in enumerate(_VIEWS):
        ax = axes_geo[row_idx][col_idx]

        for is_rs, col in [(True, _C_RS), (False, _C_NS)]:
            mask = is_rs_arr == is_rs
            if mask.any():
                ax.add_collection3d(Line3DCollection(segs[mask], color=col, lw=1.5, alpha=1.0))

        for mask, col, ms in [(_sup_v, _C_SUP, 55), (_bot_v, _C_BOT, 20), (_top_v, _C_LOAD, 20)]:
            if mask.any():
                ax.scatter3D(_vxyz[mask, 0], _vxyz[mask, 1], _vxyz[mask, 2],
                             c=col, s=ms, zorder=5)

        ax.auto_scale_xyz(vdf["x"].values, vdf["y"].values, vdf["z"].values)
        ax.set_xlabel("x [m]", fontsize=7)
        ax.set_ylabel("y [m]", fontsize=7)
        ax.set_zlabel("z [m]" if azim != -90 else "", fontsize=7)
        ax.set_box_aspect((1, 0.65, 0.28))
        ax.view_init(elev=elev, azim=azim)
        ax.tick_params(labelsize=6)

        if row_idx == 0:
            ax.set_title(
                f"GA_{label}  ·  {view_lbl}\n"
                f"reuse={bd['reuse_rate']:.1%}  ({rs_count}/120 RS)  ·  "
                f"cost={bd['total_cost']:.1f}  ·  fit={bd['fitness']:.4f}",
                fontsize=8.5, fontweight="bold", pad=8,
            )
        else:
            ax.set_title(view_lbl, fontsize=8, pad=4)

_leg = [
    mpatches.Patch(color=_C_RS,   label="Reclaimed (RS)"),
    mpatches.Patch(color=_C_NS,   label="New stock (NS)"),
    plt.Line2D([], [], marker="o", color="w", markerfacecolor=_C_SUP,  markersize=8, label="Support"),
    plt.Line2D([], [], marker="o", color="w", markerfacecolor=_C_LOAD, markersize=6, label="Top load node"),
    plt.Line2D([], [], marker="o", color="w", markerfacecolor=_C_BOT,  markersize=6, label="Bottom node"),
]
axes_geo[0][0].legend(handles=_leg, loc="upper left", fontsize=7.5,
                      framealpha=0.9, bbox_to_anchor=(-0.06, 1.22))

plt.tight_layout()
plt.show()